In [1]:
import numpy as np
import torch
from torch import nn
import torch.optim as optim
import matplotlib.pyplot as plt

%matplotlib inline
torch.set_printoptions(sci_mode=False,precision=4)
torch.manual_seed(123)
torch.random.manual_seed(123)
generator = torch.Generator().manual_seed(123)

In [2]:
context_length = 25
batch_size = 8
embd_dim = 24
iteration_count = 900
head_size = 8

In [3]:
with open('../data/numeric_data.txt', 'r', encoding = 'utf-8') as file:
    records = file.read()

In [4]:
data = ''.join(records)[0:5000]
data = '_' * context_length + data
unique_text = sorted(list(set(data)))
vocab_size = len(unique_text)
print(vocab_size)

47


In [5]:
stoi = {text: index for index, text in enumerate(unique_text)}
itos = {index: text for index, text in enumerate(unique_text)}

encoder = lambda char_array: [stoi[char] for char in ''.join(char_array)]
decoder = lambda num_array: [itos[num] for num in num_array]

In [6]:
x_numpy = []
y_numpy = []
temp_data = data

for t in range(len(temp_data) - context_length - 1):
    x_numpy.append(encoder(temp_data[t : t + context_length]))
    y_numpy.append(stoi[temp_data[t + context_length]])

x = torch.tensor(x_numpy, dtype=torch.long)
y = torch.tensor(y_numpy, dtype=torch.long)

N = x.shape[0]
train_ratio = 0.8
val_ratio = 0.1

train_end = int(train_ratio * N)
val_end = int((train_ratio + val_ratio) * N)

x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:], y[train_end:]

In [7]:
print(vocab_size, embd_dim)
criterion = nn.CrossEntropyLoss()
embedding = nn.Embedding(vocab_size, embd_dim)
logit_layer = nn.Linear(context_length * embd_dim, vocab_size)

embedding = nn.Embedding(vocab_size, embd_dim) # Assuming vocab_size is defined
key = nn.Linear(embd_dim, head_size, bias=False)
query = nn.Linear(embd_dim, head_size, bias=False)
value = nn.Linear(embd_dim, head_size, bias=False)
logit_layer = nn.Linear(head_size * context_length, vocab_size) # Adjusted input size

47 24


In [8]:
def evaluate(x_data, y_data):
    # Set all layers to evaluation mode
    embedding.eval()
    key.eval()
    query.eval()
    value.eval()
    logit_layer.eval()

    with torch.no_grad():
        # 1. Embedding
        x = embedding(x_data) # (B, T, Embd_Size)

        # 2. Self-Attention (Must match training logic exactly)
        k = key(x)   # (B, T, head_size)
        q = query(x) # (B, T, head_size)

        # Calculate attention scores
        # Note: 'tril' should be accessible globally or passed in
        T = x.size(1)
        wei = (q @ k.transpose(-2, -1)) * (k.size(-1)**-0.5)
        # Use a local mask in case T varies
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        wei = wei.masked_fill(mask == 0, float('-inf'))
        wei = torch.nn.functional.softmax(wei, dim=-1)

        # Apply weights to values
        v = value(x)
        out = wei @ v # (B, T, head_size)

        # 3. Flatten and get Logits
        # Ensure this view matches the shape used in training
        out_flattened = out.reshape(out.size(0), -1)
        logits = logit_layer(out_flattened)

        # 4. Metrics
        val_loss = criterion(logits, y_data)
        predictions = torch.argmax(logits, dim=1) # softmax not needed for argmax
        accuracy = (predictions == y_data).float().mean()

    # Set all layers back to training mode
    embedding.train()
    key.train()
    query.train()
    value.train()
    logit_layer.train()

    return val_loss.item(), accuracy.item()

In [9]:
# --- Optimizer ---
# Note: embedding and logit_layer should be in .train() mode for training
optimizer = optim.Adam(
    list(embedding.parameters()) +
    list(logit_layer.parameters()) +
    list(key.parameters()) +
    list(query.parameters()) +
    list(value.parameters()),
    lr=0.001
)

tril = torch.tril(torch.ones(context_length, context_length))
for epoch in range(iteration_count):
    # 1. Data Fetching
    idx = torch.randint(0, len(x_train), (batch_size,), generator=generator)
    rand_x = x_train[idx] # Shape: (batch, context_length)
    rand_y = y_train[idx]
    # print('rand_y : ', rand_y.shape)

    optimizer.zero_grad(set_to_none=True)

    # 2. Embedding
    x = embedding(rand_x) # Shape: (batch, context_length, embd_size)

    # 3. Self-Attention Mechanism
    k = key(x)   # (B, T, head_size)
    q = query(x) # (B, T, head_size)

    # print('x.size() : ', x.size())
    # print('q.size() : ', q.size())
    # print('k.size() : ', k.size())
    T = x.size(1)
    # Calculate attention scores (affinities)
    wei = q @ k.transpose(-2, -1) * (head_size**-0.5) # Scaled dot-product
    # wei = q @ k.transpose(-1, 0) * (head_size**-0.5) # Scaled dot-product
    # print('wei (1) : ', wei)

    wei = wei.masked_fill(tril == 0, float('-inf'))

    # print('wei (2) : ', wei)

    wei = torch.nn.functional.softmax(wei, dim=-1)

    # print('wei (3) : ', wei)

    # print('wei.size() : ', wei.size())

    # Apply weights to values
    v = value(x) # (B, T, head_size)
    out = wei @ v # (B, T, head_size)

    # print('out.size() : ', out.size())
    # out = wei @ x # (B, T, head_size)

    # 4. Preparation for Logit Layer
    # Flatten the sequence: (B, T, H) -> (B, T*H)
    out_flattened = out.view(batch_size, -1)

    # print('out_flattened.shape : ', out_flattened.shape)

    logits = logit_layer(out_flattened)
    # print('logits.shape : ', logits.shape)
    loss = criterion(logits, rand_y)

    # 5. Backprop
    if epoch % 10000 == 0:
        print(loss.item(), evaluate(x_val, y_val)[0])

    loss.backward()
    optimizer.step()

3.7579550743103027 3.883751630783081


In [10]:
# # context_length = 2
# tril = torch.tril(torch.ones(context_length, context_length))
#
# mat1 = torch.tensor([[.1,.34],[.15,.63]], dtype=torch.float)
# print('mat1.shape : ', mat1)
#
# mat2 = torch.tensor([[.85,.25],[.52,.67]], dtype=torch.float)
# print('mat2.shape : ', mat2.shape)
#
# combine = (mat1 @ mat2.t())#* (head_size**-0.5)
# print('combine : ', combine)
#
# wei = combine.masked_fill(tril == 0, float('-inf'))
# print('wei : ', wei)
#
# soft = torch.nn.functional.softmax(wei, dim=-1)
# print('soft : ', soft)
# #ROHIT

In [11]:
#Test
logging = True
final_text = 'You are all resolved rather to die than'.lower()
for _ in range(100):
        x = torch.tensor(encoder(final_text[-context_length:]))
        x = torch.unsqueeze(x, 0)
        x = embedding(x) # (B, T, Embd_Size)
        print(x)

        # 2. Self-Attention (Must match training logic exactly)
        k = key(x)   # (B, T, head_size)
        q = query(x) # (B, T, head_size)

        # Calculate attention scores
        # Note: 'tril' should be accessible globally or passed in
        T = x.size(1)
        wei = (q @ k.transpose(-2, -1)) * (k.size(-1)**-0.5)
        # Use a local mask in case T varies
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        wei = wei.masked_fill(mask == 0, float('-inf'))
        wei = torch.nn.functional.softmax(wei, dim=-1)

        # Apply weights to values
        v = value(x)
        out = wei @ v # (B, T, head_size)

        # 3. Flatten and get Logits
        # Ensure this view matches the shape used in training
        out_flattened = out.reshape(out.size(0), -1)
        logits = logit_layer(out_flattened)
        final = torch.max(logits, 1)
        y = decoder([final[1].item()])
        final_text+=y[0]


print(final_text)

tensor([[[-0.1122,  0.2632, -1.2646, -0.0245, -1.2757, -1.3341, -0.3036,
          -0.0201, -0.4324,  0.6274, -0.1088,  1.2474,  1.3790, -0.2485,
          -0.8115,  0.5721, -1.0673,  0.3209, -1.2759,  0.2399,  0.4214,
           0.4687, -0.1040, -0.5618],
         [ 1.4210,  0.3039, -0.4337, -2.2003,  0.3319, -1.1737, -0.4428,
          -1.0038,  0.8740, -1.5405, -1.0227,  0.8545, -0.0504,  0.5735,
          -0.7850, -2.6311,  0.0508, -0.5311,  0.3138, -1.3364,  0.0542,
          -0.8904,  0.7775, -0.4266],
         [-0.8226,  0.5688,  1.6902, -1.0289, -0.0603,  2.1357,  2.0908,
          -1.0313, -1.2499, -1.8621,  1.5816,  1.1882,  1.3923, -0.3740,
           0.5689,  0.0174,  0.8725, -0.1775, -0.7322, -2.0732, -0.0010,
           0.3861,  0.4759,  2.3373],
         [-0.1353,  0.2451,  0.5250, -0.6529, -2.1083, -1.5460, -0.5402,
           0.7347,  0.1377,  1.6543, -2.5964, -1.2055, -0.8208,  0.4338,
          -1.0791, -0.4435, -0.0800,  0.1129, -0.6359,  0.8349,  0.5434,
          